In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
orders_df = spark.read.table("dbt_adv_pr.silver.order_2025_all")
cust_df = spark.read.table("dbt_adv_pr.silver.customer")

ref_date = orders_df.select(F.max("order_date")).collect()[0][0]

rfm_raw = (
    orders_df
    .groupBy("customer_id")
    .agg(
        F.datediff(F.lit(ref_date), F.max("order_date")).alias("recency_days"),
        F.countDistinct("order_id").alias("frequency"),
        F.round(F.sum("amount"), 2).alias("monetary")
    )
)

r_win = Window.orderBy(F.col("recency_days").desc())
f_win = Window.orderBy(F.col("frequency").asc())
m_win = Window.orderBy(F.col("monetary").asc())

rfm_scored = (
    rfm_raw
    .withColumn("r_score", F.ntile(5).over(r_win))
    .withColumn("f_score", F.ntile(5).over(f_win))
    .withColumn("m_score", F.ntile(5).over(m_win))
    .withColumn("rfm_code", F.concat_ws("", F.col("r_score"), F.col("f_score"), F.col("m_score")))
    .withColumn(
        "segment",
        F.when((F.col("r_score") >= 4) & (F.col("f_score") >= 4) & (F.col("m_score") >= 4), "Champions")
         .when((F.col("r_score") <= 2) & ((F.col("f_score") >= 3) | (F.col("m_score") >= 3)), "At-Risk")
         .when((F.col("r_score") == 1) & (F.col("f_score") <= 2) & (F.col("m_score") <= 2), "Lost")
         .when((F.col("r_score") >= 3) & (F.col("f_score") >= 3), "Loyal Customers")
         .when(F.col("r_score") >= 4, "Recent / Promising")
         .otherwise("Hibernating / Needs Attention")
    )
)

(
    rfm_scored
    .groupBy("segment")
    .agg(
        F.count("customer_id").alias("customer_count"),
        F.round(F.avg("recency_days"), 1).alias("avg_recency_days"),
        F.round(F.avg("frequency"), 1).alias("avg_orders"),
        F.round(F.sum("monetary"), 2).alias("total_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
    .show(truncate=False)
)

target_segments = (
    rfm_scored
    .filter(F.col("segment").isin("Champions", "At-Risk", "Lost"))
    .join(cust_df, on="customer_id", how="left")
    .orderBy(F.col("segment"), F.col("monetary").desc())
)

target_segments.show(30, truncate=False)


In [ ]:
rfm_scored.write.mode('append')\
                .saveAsTable('dbt_adv_pr.gold.segment_scorecard')

In [ ]:
target_segments.write.mode('append')\
                .saveAsTable('dbt_adv_pr.gold.customer_segment_scorecard')

In [ ]:
orders_df = spark.read.table("dbt_adv_pr.silver.order_2025_all")
cust_df = spark.read.table("dbt_adv_pr.silver.customer")

bounds = orders_df.select(
    F.min("order_date").alias("min_dt"),
    F.max("order_date").alias("max_dt")
).collect()[0]

total_days = (bounds["max_dt"] - bounds["min_dt"]).days + 1
years_observed = total_days / 365.0

cust_metrics = (
    orders_df
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("amount"), 2).alias("historical_clv"),
        F.min("order_date").alias("first_order_date"),
        F.max("order_date").alias("last_order_date")
    )
    .withColumn("aov", F.round(F.col("historical_clv") / F.col("total_orders"), 2))
    .withColumn("annual_frequency", F.round(F.col("total_orders") / F.lit(years_observed), 2))
)

avg_lifespan_years = 2.5
gross_margin = 1.0

clv_df = (
    cust_metrics
    .withColumn(
        "predictive_clv",
        F.round(F.col("aov") * F.col("annual_frequency") * F.lit(avg_lifespan_years) * F.lit(gross_margin), 2)
    )
    .join(cust_df, on="customer_id", how="left")
    .orderBy(F.col("predictive_clv").desc())
)

clv_df.show(20, truncate=False)

(
    clv_df
    .agg(
        F.count("customer_id").alias("total_customers"),
        F.round(F.avg("aov"), 2).alias("avg_order_value"),
        F.round(F.avg("annual_frequency"), 2).alias("avg_orders_per_year"),
        F.round(F.avg("historical_clv"), 2).alias("avg_historical_clv"),
        F.round(F.expr("percentile_approx(historical_clv, 0.5)"), 2).alias("median_historical_clv"),
        F.round(F.avg("predictive_clv"), 2).alias("avg_predictive_clv")
    )
    .show(truncate=False)
)


In [ ]:
clv_df.write.mode('append')\
            .saveAsTable("dbt_adv_pr.gold.customer_lv")

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.functions import vector_to_array

orders_df = spark.read.table("dbt_adv_pr.silver.order_2025_all")

max_date = orders_df.select(F.max("order_date")).collect()[0][0]
churn_window_days = 60
cutoff_date = orders_df.select(F.date_sub(F.lit(max_date), churn_window_days)).collect()[0][0]

hist_features = (
    orders_df
    .filter(F.col("order_date") <= F.lit(cutoff_date))
    .groupBy("customer_id")
    .agg(
        (F.datediff(F.max("order_date"), F.min("order_date")) + 1).alias("active_days"),
        F.datediff(F.lit(cutoff_date), F.max("order_date")).alias("no_shopping")
    )
)

future_buyers = (
    orders_df
    .filter(F.col("order_date") > F.lit(cutoff_date))
    .select("customer_id")
    .distinct()
    .withColumn("retained", F.lit(1))
)

train_df = (
    hist_features.join(future_buyers, on="customer_id", how="left")
    .withColumn("label", F.when(F.col("retained").isNull(), 1).otherwise(0))
)

assembler = VectorAssembler(inputCols=["active_days", "no_shopping"], outputCol="features")
train_vec = assembler.transform(train_df)

lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train_vec)

current_df = (
    orders_df
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        (F.datediff(F.max("order_date"), F.min("order_date")) + 1).alias("active_days"),
        F.datediff(F.lit(max_date), F.max("order_date")).alias("no_shopping")
    )
)

current_vec = assembler.transform(current_df)

churn_predictions = (
    lr_model.transform(current_vec)
    .withColumn("churn_probability", F.round(vector_to_array(F.col("probability"))[1], 4))
    .withColumn(
        "churn_risk_tier",
        F.when(F.col("churn_probability") >= 0.75, "High Risk")
         .when(F.col("churn_probability") >= 0.40, "Medium Risk")
         .otherwise("Low Risk")
    )
    .select("customer_id", "total_orders", "active_days", "no_shopping", "churn_probability", "churn_risk_tier")
    .orderBy(F.col("churn_probability").desc())
)

churn_predictions.show(20, truncate=False)


In [ ]:
churn_predictions.write.mode('append')\
                       .saveAsTable('dbt_adv_pr.gold.customer_churn_predictions')

In [ ]:
from pyspark.sql import functions as F

orders_df = spark.read.table("dbt_adv_pr.silver.order_2025_all")

# 1. Deduplicate items per order (in case the same SKU appears on multiple lines in one order)
basket_items = (
    orders_df
    .select("order_id", "sku_id", "item_name")
    .distinct()
)

# 2. Total number of distinct orders (baskets)
total_orders = basket_items.select("order_id").distinct().count()

# 3. Individual item order counts and standalone support
item_counts = (
    basket_items
    .groupBy("sku_id", "item_name")
    .agg(F.count("order_id").alias("item_orders"))
    .withColumn("item_support", F.col("item_orders") / F.lit(total_orders))
)

# 4. Pairwise co-occurrence within the same order_id (directional A -> B)
pairs_df = (
    basket_items.alias("a")
    .join(
        basket_items.alias("b"),
        (F.col("a.order_id") == F.col("b.order_id")) & (F.col("a.sku_id") != F.col("b.sku_id")),
        how="inner"
    )
    .groupBy(
        F.col("a.sku_id").alias("sku_a"),
        F.col("a.item_name").alias("item_a"),
        F.col("b.sku_id").alias("sku_b"),
        F.col("b.item_name").alias("item_b")
    )
    .agg(F.countDistinct("a.order_id").alias("pair_orders"))
    .filter(F.col("pair_orders") >= 5)  # Filter out rare noise pairs; adjust threshold as needed
)

# 5. Join individual stats to compute Support, Confidence, and Lift
mba_rules = (
    pairs_df
    .join(
        item_counts.select(
            F.col("sku_id").alias("sku_a"),
            F.col("item_orders").alias("orders_a")
        ),
        on="sku_a",
        how="inner"
    )
    .join(
        item_counts.select(
            F.col("sku_id").alias("sku_b"),
            F.col("item_support").alias("support_b")
        ),
        on="sku_b",
        how="inner"
    )
    .withColumn("support_pct", F.round((F.col("pair_orders") / F.lit(total_orders)) * 100, 2))
    .withColumn("confidence_pct", F.round((F.col("pair_orders") / F.col("orders_a")) * 100, 2))
    .withColumn("lift", F.round((F.col("pair_orders") / F.col("orders_a")) / F.col("support_b"), 2))
    .filter(F.col("lift") > 1.0)
    .select(
        "sku_a",
        "item_a",
        "sku_b",
        "item_b",
        "pair_orders",
        "support_pct",
        "confidence_pct",
        "lift"
    )
    .orderBy(F.col("lift").desc(), F.col("pair_orders").desc())
)

# Show top 25 product affinity rules
mba_rules.show(25, truncate=False)


In [ ]:
mba_rules.write.mode('append')\
               .saveAsTable('dbt_adv_pr.gold.market_analysis') 